# 🔍 Job Advert Scraper — hiring.cafe

This notebook automates the extraction of job listings from [hiring.cafe](https://hiring.cafe) into a structured CSV database.

It uses **Selenium with undetected-chromedriver** to navigate the site like a real user, open each job card, and extract key fields:
- Job Title, Company, Salary, Position Type, Remote status
- Responsibilities, Requirements Summary, Technical Tools

---

### Workflow Overview
```
1. CONFIGURE  →  Set search URL & output file path
2. LAUNCH     →  Start browser (undetected Chrome)
3. SCROLL     →  Lazy-scroll the results page to load all cards
4. SCRAPE     →  Loop through each job card, open Full View, extract data
5. SAVE       →  Append each job row to CSV
6. REVIEW     →  Preview and inspect the collected data
```

---

> **Tip:** Run cells one section at a time. Each section is independently runnable so you can re-run, debug, or tweak specific parts without restarting the whole flow.

---
## 📦 Section 1 — Imports & Dependencies

All required libraries are imported here.

**Prerequisites** — install these if not already present:
```bash
pip install selenium undetected-chromedriver pandas
```
- `selenium` — browser automation framework
- `undetected_chromedriver` — bypasses bot-detection on modern sites
- `pandas` — tabular data handling and CSV I/O
- `time`, `os`, `pathlib` — standard library utilities

In [ ]:
import time
import os
from pathlib import Path

import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver as uc

print("✅ All imports successful.")

---
## ⚙️ Section 2 — Configuration

**This is the primary cell you'll edit between scraping sessions.**

| Parameter | Description |
|---|---|
| `SEARCH_URL` | The full hiring.cafe URL with your search filters applied |
| `OUTPUT_CSV` | Path where job rows are saved (appended on each run) |
| `PAUSE_TIME` | Seconds to wait after each scroll increment (increase if jobs load slowly) |
| `MAX_SCROLL_ATTEMPTS` | Safety cap on scroll iterations |
| `PAGE_LOAD_WAIT` | Seconds to wait for the page to initially load |

### How to get your search URL
1. Go to [hiring.cafe](https://hiring.cafe)
2. Enter your search query, date range, and location filters
3. Copy the full URL from your browser's address bar and paste it below

In [ ]:
# ── Search URL ────────────────────────────────────────────────────────────────
# Paste your hiring.cafe search URL here.
# The URL encodes all your filters (query, location, date range, etc.)
SEARCH_URL = (
    "https://hiring.cafe/?searchState=%7B%22searchQuery%22%3A%22marketing+director%22"
    "%2C%22dateFetchedPastNDays%22%3A14%2C%22locations%22%3A%5B%7B%22id%22%3A%22ZhY1yZQBoEtHp_8UErzY%22"
    "%2C%22types%22%3A%5B%22administrative_area_level_1%22%5D%2C%22address_components%22%3A%5B%7B%22long_name%22"
    "%3A%22New+York%22%2C%22short_name%22%3A%22NY%22%2C%22types%22%3A%5B%22administrative_area_level_1%22%5D%7D"
    "%2C%7B%22long_name%22%3A%22United+States%22%2C%22short_name%22%3A%22US%22%2C%22types%22%3A%5B%22country%22"
    "%5D%7D%5D%2C%22formatted_address%22%3A%22New+York%2C+United+States%22%2C%22population%22%3A19274244"
    "%2C%22workplace_types%22%3A%5B%5D%2C%22options%22%3A%7B%22flexible_regions%22%3A%5B%22anywhere_in_country%22"
    "%2C%22anywhere_in_continent%22%2C%22anywhere_in_world%22%5D%7D%7D%5D%7D"
)

# ── Output file ───────────────────────────────────────────────────────────────
# CSV file where scraped jobs will be saved.
# If this file already exists, new rows are appended to it (no duplicates check built-in).
OUTPUT_CSV = Path("jobs.csv")

# ── Scroll settings ───────────────────────────────────────────────────────────
# Increase PAUSE_TIME on slow connections or if job cards aren't loading fully.
PAUSE_TIME = 5          # seconds to pause after each full scroll cycle
MAX_SCROLL_ATTEMPTS = 40  # max scroll cycles before giving up

# ── Page load wait ────────────────────────────────────────────────────────────
PAGE_LOAD_WAIT = 5      # seconds to wait for the initial page to render

print(f"✅ Configuration loaded.")
print(f"   Output file : {OUTPUT_CSV.resolve()}")
print(f"   Pause time  : {PAUSE_TIME}s | Max scrolls: {MAX_SCROLL_ATTEMPTS}")

---
## 🌐 Section 3 — Browser Setup

Launches an **undetected Chrome** instance configured to:
- Disable GPU rendering (avoids headless-mode rendering bugs)
- Open at 1920×1080 to ensure all page elements render as on a normal desktop
- Suppress Selenium's automation flags so the site doesn't block the scraper

> **Note:** A real Chrome browser window will open on your screen — this is expected and required. Do not close it manually during the scrape.

In [ ]:
# Configure Chrome options
options = uc.ChromeOptions()
options.add_argument('--disable-gpu')                          # Prevents GPU-related crashes
options.add_argument('--window-size=1920,1080')                # Full desktop resolution
options.add_argument('--disable-blink-features=AutomationControlled')  # Hides automation flag

# Launch the browser
driver = uc.Chrome(options=options)

print("✅ Browser launched successfully.")
print("   A Chrome window should now be visible on your screen.")

---
## 🖱️ Section 4 — Scroll Helper Function

hiring.cafe uses **lazy loading** — job cards only render as you scroll down. This function simulates a real user scrolling through the page in increments to trigger all cards to load before scraping begins.

**How it works:**
1. Scrolls to 25%, 50%, 75%, and 100% of the current page height
2. Waits for new content to load
3. Checks if the page height grew (new cards rendered)
4. Repeats until the page height stops growing or `max_attempts` is reached

In [ ]:
def scroll_to_load_all_jobs(driver, pause_time=PAUSE_TIME, max_attempts=MAX_SCROLL_ATTEMPTS):
    """
    Incrementally scroll the page to trigger lazy-loading of all job cards.

    Parameters
    ----------
    driver        : Selenium WebDriver instance
    pause_time    : Seconds to wait after each full scroll cycle (default: PAUSE_TIME)
    max_attempts  : Maximum number of scroll cycles before stopping (default: MAX_SCROLL_ATTEMPTS)

    Returns
    -------
    None — scrolls in-place, all loaded cards become available in the DOM
    """
    last_height = driver.execute_script("return document.body.scrollHeight")
    attempts = 0

    while attempts < max_attempts:
        # Scroll in four incremental steps to simulate natural user behaviour
        for frac in [0.25, 0.5, 0.75, 1.0]:
            driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{frac});")
            time.sleep(0.5)  # Small pause between increments

        # Final scroll to absolute bottom
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause_time)  # Wait for new cards to render

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            # Height unchanged — wait a bit longer in case of slow network
            time.sleep(3)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                print(f"   ↳ No new content after attempt {attempts + 1}. All cards likely loaded.")
                break  # Page is fully loaded
            last_height = new_height
        else:
            last_height = new_height  # Page grew — continue scrolling

        attempts += 1

    print(f"✅ Scroll complete. Total scroll cycles: {attempts}")


print("✅ scroll_to_load_all_jobs() defined.")

---
## 🗂️ Section 5 — Data Extraction Helper Function

This function extracts all structured fields from a single job's **Full View** page.

Each field has its own `try/except` block so that:
- If one element is missing or the selector breaks, only that field is left empty
- The rest of the job's data is still saved normally
- The scraper keeps running rather than crashing

**Fields extracted:**

| Field | Description |
|---|---|
| `URL` | Direct link to the job's full view page |
| `JOB TITLE` | Role title (e.g. "Marketing Director") |
| `COMPANY` | Employer name |
| `SALARY` | Salary range if shown (e.g. "$120k–$150k /yr"), else "N/A" |
| `JOB POSITION` | Employment type (e.g. "Full Time") |
| `REMOTE` | "Yes" if remote label is present, else "No" |
| `RESPONSIBILITIES` | Bullet-point summary of role responsibilities |
| `REQUIREMENTS SUMMARY` | Key qualifications and experience requirements |
| `TECHNICAL TOOLS MENTIONED` | Software/tools explicitly named in the listing |

In [ ]:
def extract_job_data(driver):
    """
    Extract structured fields from the currently open job Full View page.

    Each field extraction is wrapped in its own try/except so a missing element
    only blanks that field rather than crashing the entire scrape.

    Parameters
    ----------
    driver : Selenium WebDriver instance (must already be on the job Full View tab)

    Returns
    -------
    dict : Upper-cased field names mapped to extracted string values
    """
    job_data = {}

    # ── URL ──────────────────────────────────────────────────────────────────
    job_data['url'] = driver.current_url

    # ── Job Title ─────────────────────────────────────────────────────────────
    try:
        job_title = driver.find_element(
            By.CSS_SELECTOR, 'h2.font-extrabold.text-3xl.text-gray-800.mb-4'
        ).text
    except Exception:
        job_title = ''  # Element not found or text unavailable
    job_data['job title'] = job_title

    # ── Company Name ──────────────────────────────────────────────────────────
    try:
        company = driver.find_element(
            By.CSS_SELECTOR, 'span.text-xl.font-semibold.text-gray-700.flex-none'
        ).text
        # Strip leading "@ " or "@" prefix that the site prepends to company names
        if company.startswith('@ '):
            company = company[2:]
        elif company.startswith('@'):
            company = company[1:]
    except Exception:
        company = ''
    job_data['company'] = company

    # ── Salary ────────────────────────────────────────────────────────────────
    # Looks for a badge element containing "/yr" — salary is not always shown
    try:
        salary = driver.find_element(
            By.XPATH,
            "//span[contains(@class, 'rounded') and contains(@class, 'font-bold') and contains(text(), '/yr')]"
        ).text
    except Exception:
        salary = ''
    job_data['salary'] = salary if salary.strip() else 'N/A'  # Default to N/A if empty

    # ── Job Position Type ─────────────────────────────────────────────────────
    # Extracts the employment type badge, e.g. "Full Time", "Contract"
    try:
        job_position = driver.find_element(
            By.XPATH,
            "//span[contains(@class, 'rounded') and contains(@class, 'font-bold') and text()='Full Time']"
        ).text
    except Exception:
        job_position = ''
    job_data['Job position'] = job_position

    # ── Remote Status ─────────────────────────────────────────────────────────
    # Presence of the "Remote" badge is treated as a boolean Yes/No
    try:
        driver.find_element(
            By.XPATH,
            "//span[contains(@class, 'rounded') and contains(@class, 'font-bold') and text()='Remote']"
        )
        job_data['Remote'] = 'Yes'
    except Exception:
        job_data['Remote'] = 'No'

    # ── Responsibilities ──────────────────────────────────────────────────────
    try:
        responsibilities = driver.find_element(
            By.XPATH,
            "//div[contains(@class, 'flex') and contains(@class, 'flex-col')"
            " and .//span[contains(text(), 'Responsibilities:')]]/span[2]"
        ).text
    except Exception:
        responsibilities = ''
    job_data['Responsibilities'] = responsibilities

    # ── Requirements Summary ──────────────────────────────────────────────────
    try:
        requirements = driver.find_element(
            By.XPATH,
            "//div[contains(@class, 'flex') and contains(@class, 'flex-col')"
            " and contains(@class, 'space-y-3')"
            " and .//span[contains(@class, 'font-bold') and contains(text(), 'Requirements Summary:')]]/span[2]"
        ).text
    except Exception:
        requirements = ''
    job_data['Requirements Summary'] = requirements

    # ── Technical Tools ───────────────────────────────────────────────────────
    try:
        tools = driver.find_element(
            By.XPATH,
            "//div[contains(@class, 'flex') and contains(@class, 'flex-col')"
            " and contains(@class, 'space-y-3')"
            " and .//span[contains(@class, 'font-bold') and contains(text(), 'Technical Tools Mentioned:')]]/span[2]"
        ).text
    except Exception:
        tools = ''
    job_data['Technical Tools Mentioned'] = tools

    # ── Uppercase all keys for consistent CSV column names ────────────────────
    return {k.upper(): v for k, v in job_data.items()}


print("✅ extract_job_data() defined.")

---
## 💾 Section 6 — CSV Save Helper Function

Handles **incremental saving** — each job is appended to the CSV as it's scraped, so if the scraper is interrupted mid-run, previously collected data is not lost.

If the output file doesn't yet exist, it's created from scratch.

In [ ]:
def save_job_to_csv(job_data: dict, output_path: Path):
    """
    Append a single job's data to the output CSV file.

    If the file already exists, the new row is concatenated to the existing data.
    If the file does not exist yet, it is created fresh.

    Parameters
    ----------
    job_data    : dict — upper-cased field names and their scraped values
    output_path : Path — file path for the output CSV

    Returns
    -------
    None
    """
    new_row = pd.DataFrame([job_data])

    if output_path.exists():
        # Append to existing file
        existing = pd.read_csv(output_path)
        df = pd.concat([existing, new_row], ignore_index=True)
    else:
        # First job — create the file
        df = new_row

    df.to_csv(output_path, index=False, na_rep='N/A')


print("✅ save_job_to_csv() defined.")

---
## 🚀 Section 7 — Main Scrape Loop

This is the core execution cell. It runs the full scraping workflow:

1. **Navigate** to the search URL
2. **Scroll** to load all job cards into the DOM
3. **Loop** through each job card:
   - Click the card to open the quick-view modal
   - Find and click the **Full View** link (opens in a new tab)
   - Extract all fields from the full job page
   - Save the row to CSV
   - Close the tab and return to the results page
   - Close the modal
4. **Quit** the browser cleanly when done

> ⚠️ **Do not interact with the browser window while this cell is running.** Selenium controls the mouse and keyboard — any interference may cause clicks to fail.

In [ ]:
# Track overall results for the summary at the end
scraped_count = 0
failed_count = 0
failed_cards = []  # Stores (index, reason) for any cards that couldn't be processed

try:
    # ── Step 1: Navigate to the search results page ───────────────────────────
    print(f"🌐 Loading search URL...")
    driver.get(SEARCH_URL)
    time.sleep(PAGE_LOAD_WAIT)  # Allow the page's JavaScript to fully render
    print("   Page loaded.")

    # ── Step 2: Scroll to trigger lazy-loading of all job cards ───────────────
    print("\n🖱️ Scrolling to load all job cards...")
    scroll_to_load_all_jobs(driver)

    # ── Step 3: Locate all job card elements in the DOM ───────────────────────
    print("\n🔎 Locating job cards...")
    job_card_selector = (
        'div.relative.flex.flex-col.items.start.w-full.rounded-x-lg.rounded-t-lg'
    )
    job_cards = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, job_card_selector))
    )
    total_cards = len(job_cards)
    print(f"   Found {total_cards} job card(s). Starting extraction...\n")

    # ── Step 4: Iterate through each job card ─────────────────────────────────
    for idx, card in enumerate(job_cards):
        card_label = f"Card {idx + 1}/{total_cards}"
        print(f"{'─'*55}")
        print(f"[{card_label}] Processing...")

        try:
            # Bring the card into the viewport before clicking
            driver.execute_script(
                "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", card
            )
            time.sleep(0.5)

            # Use ActionChains for reliable click simulation
            ActionChains(driver).move_to_element(card).pause(0.5).click(card).perform()
            print(f"   ✓ Clicked card. Waiting for modal...")

            # ── Wait for the quick-view modal to appear ────────────────────────
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, '[role="dialog"], [aria-modal="true"]')
                    )
                )
                print("   ✓ Modal opened.")

                # ── Find the modal header ──────────────────────────────────────
                header = WebDriverWait(driver, 10).until(
                    EC.visibility_of_element_located(
                        (By.CSS_SELECTOR, 'header.chakra-modal__header')
                    )
                )

                # ── Locate and click the 'Full View' link inside the header ────
                try:
                    full_view_link = header.find_element(
                        By.XPATH,
                        ".//a[span[text()='Full View'] and starts-with(@href, 'https://hiring.cafe/job/')]"
                    )
                    job_url = full_view_link.get_attribute('href')
                    time.sleep(1.5)  # Brief pause before clicking to avoid mis-clicks
                    full_view_link.click()
                    print(f"   ✓ Full View link clicked → {job_url}")

                    # ── Switch focus to the newly opened tab ──────────────────
                    time.sleep(2)  # Allow the new tab to start loading
                    driver.switch_to.window(driver.window_handles[-1])
                    print("   ✓ Switched to Full View tab.")

                    # ── Extract all job fields from the Full View page ─────────
                    job_data = extract_job_data(driver)

                    # ── Save the extracted row to CSV ──────────────────────────
                    save_job_to_csv(job_data, OUTPUT_CSV)
                    scraped_count += 1
                    print(f"   ✅ Saved: {job_data.get('JOB TITLE', 'Unknown')} @ {job_data.get('COMPANY', 'Unknown')}")

                    # ── Close the Full View tab and return to results ──────────
                    driver.close()
                    driver.switch_to.window(driver.window_handles[0])
                    print("   ✓ Closed Full View tab, back on results page.")

                except Exception as e:
                    # Full View link not found — log the modal header's anchors for debugging
                    print(f"   ⚠️ Could not find 'Full View' link: {e}")
                    anchors = header.find_elements(By.TAG_NAME, 'a')
                    print(f"      Found {len(anchors)} anchor(s) in header:")
                    for a in anchors:
                        print(f"      {a.get_attribute('outerHTML')}")
                    failed_count += 1
                    failed_cards.append((idx + 1, str(e)))

            except Exception as e:
                print(f"   ⚠️ Modal did not appear: {e}")
                failed_count += 1
                failed_cards.append((idx + 1, str(e)))

            # ── Close the modal on the main tab before moving to the next card ─
            try:
                header = WebDriverWait(driver, 5).until(
                    EC.visibility_of_element_located(
                        (By.CSS_SELECTOR, 'header.chakra-modal__header')
                    )
                )
                # Find the close button(s) and click the last one (most reliable)
                close_btns = header.find_elements(
                    By.CSS_SELECTOR,
                    r'button.rounded-lg.p-2.text-black.hover\:bg-gray-200.flex-none.outline-none'
                )
                if close_btns:
                    close_btns[-1].click()
                    print("   ✓ Modal closed.")
                else:
                    print("   ⚠️ Close button not found — modal may already be gone.")
            except Exception as e:
                # Modal might have closed itself; not critical
                print(f"   ℹ️ Modal close step: {e}")

        except Exception as e:
            print(f"   ❌ Could not process card: {e}")
            failed_count += 1
            failed_cards.append((idx + 1, str(e)))

except Exception as outer_e:
    print(f"\n❌ Critical error in main loop: {outer_e}")

finally:
    # ── Always quit the browser cleanly ───────────────────────────────────────
    driver.quit()
    print("\n🔒 Browser closed.")

---
## 📊 Section 8 — Run Summary

Prints a summary of how the scrape went — useful for spotting systematic failures (e.g. all cards failing means a CSS selector may have broken).

In [ ]:
print("=" * 55)
print("  SCRAPE SUMMARY")
print("=" * 55)
print(f"  ✅ Jobs successfully scraped : {scraped_count}")
print(f"  ❌ Cards failed              : {failed_count}")
print(f"  📁 Output file               : {OUTPUT_CSV.resolve()}")

if failed_cards:
    print("\n  Failed card details:")
    for card_idx, reason in failed_cards:
        # Truncate long error messages for readability
        short_reason = (reason[:120] + '...') if len(reason) > 120 else reason
        print(f"    Card #{card_idx}: {short_reason}")

print("=" * 55)

---
## 👀 Section 9 — Preview Collected Data

Load the CSV and inspect the results. Useful to verify the scrape worked correctly and spot any malformed rows.

In [ ]:
if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV)
    print(f"Total rows in {OUTPUT_CSV.name}: {len(df)}")
    print(f"Columns: {list(df.columns)}\n")
    df.head(10)  # Show first 10 rows
else:
    print(f"⚠️ Output file not found: {OUTPUT_CSV.resolve()}")
    print("   The scrape may not have saved any rows.")

---
## 🔎 Section 10 — Data Quality Check

Quickly inspect:
- How many jobs have salary information vs N/A
- How many are remote
- Which fields have missing values

In [ ]:
if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV)

    print("── Missing / empty values per column ──────────────────")
    print(df.isnull().sum().to_string())

    print("\n── Salary availability ────────────────────────────────")
    salary_counts = df['SALARY'].value_counts(dropna=False)
    has_salary = (df['SALARY'] != 'N/A').sum()
    print(f"  Jobs with salary shown : {has_salary}")
    print(f"  Jobs without salary    : {(df['SALARY'] == 'N/A').sum()}")

    if 'REMOTE' in df.columns:
        print("\n── Remote vs On-site ───────────────────────────────────")
        print(df['REMOTE'].value_counts().to_string())

    if 'JOB POSITION' in df.columns:
        print("\n── Job Position Types ──────────────────────────────────")
        print(df['JOB POSITION'].value_counts().to_string())
else:
    print("⚠️ No data file found yet.")

---
## 🧹 Section 11 — Optional: Deduplicate the CSV

If you run the scraper multiple times on overlapping search results, the same job may appear more than once. This cell removes duplicate rows based on the job URL (which is unique per listing).

> **Safe to run at any time.** The original CSV is overwritten with the deduplicated version.

In [ ]:
if OUTPUT_CSV.exists():
    df = pd.read_csv(OUTPUT_CSV)
    original_count = len(df)

    # Drop rows where the URL is duplicated — keep the first occurrence
    df_deduped = df.drop_duplicates(subset=['URL'], keep='first')
    removed = original_count - len(df_deduped)

    df_deduped.to_csv(OUTPUT_CSV, index=False, na_rep='N/A')

    print(f"✅ Deduplication complete.")
    print(f"   Rows before : {original_count}")
    print(f"   Duplicates  : {removed}")
    print(f"   Rows after  : {len(df_deduped)}")
else:
    print("⚠️ No output file found to deduplicate.")

---
## 🔧 Section 12 — Troubleshooting Guide

Use this reference if the scraper runs but produces unexpected results.

### Job cards not loading / `No job cards found`
- Increase `PAUSE_TIME` (e.g. from 5 → 8) in the Configuration cell
- Increase `PAGE_LOAD_WAIT` (e.g. from 5 → 10) to give the page more time to render
- Check whether the CSS selector for job cards still matches by inspecting the page manually

### Modal doesn't appear after clicking a card
- The site may have changed its interaction model — verify manually that clicking a card opens a modal
- Try increasing the `time.sleep(0.5)` after scrolling the card into view

### `Full View` link not found in the header
- The XPATH for the Full View anchor (`a[span[text()='Full View']]`) may have changed
- The cell prints all anchors found in the header — use those to identify the correct selector

### Fields coming back empty
- Inspect the job's Full View page manually and use browser DevTools to find the updated selector
- Each field's selector is isolated in `extract_job_data()` — update only the affected one

### Browser crashes or `WebDriverException`
- Make sure `undetected-chromedriver` version matches your installed Chrome version: `pip install --upgrade undetected-chromedriver`
- If the browser was already open from a previous failed run, restart the kernel and re-run from Section 3

### Site blocks the scraper (Captcha / bot detection)
- Increase all `time.sleep()` values to slow down interactions
- Run the scraper in smaller batches with manual pauses between runs
- Try a different network (the site may have flagged your IP temporarily)